# Maximum-wind forecast comparison

Load Ragasa and Yagi maximum-wind forecasts, align them with the JMA best tracks, validate the expected time points, and export one two-panel figure per storm.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

INPUT_DIR = Path("./input")
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_STYLES = {
    "era5": {"label": "ERA5_WRF", "color": "#4c78a8", "marker": "x"},
    "pangu": {"label": "Pangu_WRF", "color": "#f58518", "marker": "s"},
    "graphcast": {"label": "GraphCast_WRF", "color": "#54a24b", "marker": "^"},
    "fengwu": {"label": "FengWu_WRF", "color": "#e45756", "marker": "D"},
    "fuxi": {"label": "FuXi_WRF", "color": "#72b7b2", "marker": "v"},
    "aurora": {"label": "Aurora_WRF", "color": "#b279a2", "marker": "p"},
}

STORM_CONFIG = {
    "Ragasa": {
        "maxwspd_dir": INPUT_DIR / "maxwspd_interval1h_ragasa",
        "file_template": "maxwspd_d01_interval1h_{model}_{cycle}.txt",
        "best_track_file": INPUT_DIR / "best_track_Ragasa_JMA.xlsx",
        "time_step_hours": 1,
        "init_times": {
            "24h": pd.Timestamp("2025-09-22 22:00"),
            "48h": pd.Timestamp("2025-09-21 22:00"),
        },
        "end_time": pd.Timestamp("2025-09-24 10:00"),
        "closest_time": pd.Timestamp("2025-09-23 22:00"),
        "year": 2025,
        "output_file": "maxwspd_forecast_ragasa.tif",
        "expected_raw_points": {"24h": 37, "48h": 61},
        "expected_plot_points": {"24h": 6, "48h": 10},
    },
    "Yagi": {
        "maxwspd_dir": INPUT_DIR / "maxwspd_inteerval6h_yagi",
        "file_template": "maxwspd_yagi_{model}_{cycle}.txt",
        "best_track_file": INPUT_DIR / "best_track_Yagi_JMA.xlsx",
        "time_step_hours": 6,
        "init_times": {
            "24h": pd.Timestamp("2024-09-04 12:00"),
            "48h": pd.Timestamp("2024-09-03 12:00"),
        },
        "end_time": pd.Timestamp("2024-09-06 00:00"),
        "closest_time": pd.Timestamp("2024-09-05 12:00"),
        "year": 2024,
        "output_file": "maxwspd_forecast_yagi.tif",
        "expected_raw_points": {"24h": 9, "48h": 13},
        "expected_plot_points": {"24h": 7, "48h": 11},
    },
}

STORM_ORDER = ("Ragasa", "Yagi")
CYCLES = ("24h", "48h")
KT_TO_MS = 0.514444
TIME_MARGIN = pd.Timedelta(hours=2)
Y_LABEL = r"Intensity (m s$^{-1}$)"
X_TICK_INTERVAL_HOURS = 12
X_MINOR_TICK_INTERVAL_HOURS = 6
FIG_SIZE = (8, 8)
LOWER_Y_PADDING_FRACTION = 0.18
LEGEND_FONT_SIZE = 14
PANEL_LABEL_POSITION = (0.02, 0.05)
TEXT_BBOX = {
    "facecolor": "white",
    "alpha": 0.65,
    "edgecolor": "none",
    "pad": 1.5,
}

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 16

In [ ]:
def load_maxwspd_forecast(config, storm, model, cycle):
    file_name = config["file_template"].format(model=model, cycle=cycle)
    values = np.atleast_2d(np.loadtxt(config["maxwspd_dir"] / file_name))
    lead_hour = np.arange(values.shape[0], dtype=int) * config["time_step_hours"]
    init_time = config["init_times"][cycle]

    return pd.DataFrame(
        {
            "storm": storm,
            "model": model,
            "cycle": cycle,
            "lead_hour": lead_hour,
            "init_time": init_time,
            "valid_time": init_time + pd.to_timedelta(lead_hour, unit="h"),
            "forecast_maxwind_ms": values[:, -1],
        }
    )


def load_best_track(file_path):
    return (
        pd.read_excel(file_path)
        .rename(columns={"Time": "time", "Wind": "wind_kt"})
        .assign(
            time=lambda df: pd.to_datetime(df["time"].astype(str), format="%y%m%d%H"),
            wind_kt=lambda df: pd.to_numeric(df["wind_kt"], errors="coerce"),
            wind_ms=lambda df: pd.to_numeric(df["wind_kt"], errors="coerce") * KT_TO_MS,
        )
        [["time", "wind_kt", "wind_ms"]]
        .dropna(subset=["time", "wind_ms"])
        .sort_values("time")
        .reset_index(drop=True)
    )


def prepare_cycle_series(storm, cycle, config, forecasts, best_track):
    init_time = config["init_times"][cycle]
    end_time = config["end_time"]
    best_window = (
        best_track.loc[lambda df: df["time"].between(init_time, end_time)]
        .copy()
        .sort_values("time")
        .reset_index(drop=True)
    )

    forecast_aligned = {}
    for model, forecast in forecasts.items():
        forecast_aligned[model] = (
            forecast.loc[forecast["valid_time"].between(init_time, end_time)]
            .rename(columns={"valid_time": "time"})
            .merge(best_window, on="time", how="inner")
            .sort_values("time")
            .reset_index(drop=True)
        )

    shared_times = pd.Index(best_window["time"])
    for forecast in forecast_aligned.values():
        shared_times = shared_times.intersection(pd.Index(forecast["time"]))

    best_aligned = (
        best_window.loc[best_window["time"].isin(shared_times)]
        .copy()
        .reset_index(drop=True)
    )
    forecast_aligned = {
        model: (
            forecast.loc[forecast["time"].isin(shared_times)]
            .copy()
            .reset_index(drop=True)
        )
        for model, forecast in forecast_aligned.items()
    }

    return {
        "storm": storm,
        "cycle": cycle,
        "best_track": best_aligned,
        "forecast_aligned": forecast_aligned,
    }


def compute_shared_time_limits(storm_series):
    all_times = []
    for cycle_data in storm_series.values():
        all_times.append(cycle_data["best_track"]["time"])
        all_times.extend(
            forecast["time"] for forecast in cycle_data["forecast_aligned"].values()
        )

    combined_times = pd.concat(all_times, ignore_index=True)
    return combined_times.min() - TIME_MARGIN, combined_times.max() + TIME_MARGIN

In [ ]:
def plot_cycle_panel(ax, storm, cycle, config, cycle_data, shared_time_limits):
    best_track = cycle_data["best_track"]
    model_lines = []

    for model, forecast in cycle_data["forecast_aligned"].items():
        style = MODEL_STYLES[model]
        line, = ax.plot(
            forecast["time"],
            forecast["forecast_maxwind_ms"],
            color=style["color"],
            linewidth=1.2,
            marker=style["marker"],
            markersize=5,
            label=style["label"],
        )
        model_lines.append(line)

    best_line, = ax.plot(
        best_track["time"],
        best_track["wind_ms"],
        color="black",
        linewidth=1.5,
        marker="o",
        markersize=5,
        label="Best track",
    )

    ax.axvline(
        config["closest_time"],
        color="#444444",
        linestyle="--",
        linewidth=1.0,
        zorder=1,
    )
    ax.set_ylabel(Y_LABEL)
    ax.set_xlim(shared_time_limits)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.4)
    ax.text(
        *PANEL_LABEL_POSITION,
        f"{storm}_{cycle}",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontweight="bold",
        bbox=TEXT_BBOX,
    )

    return [best_line] + model_lines


def format_time_axis(ax, year, show_xlabel):
    ax.set_xlabel(f"Time (UTC, {year})" if show_xlabel else "")
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=X_TICK_INTERVAL_HOURS))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %HZ"))
    ax.xaxis.set_minor_locator(mdates.HourLocator(interval=X_MINOR_TICK_INTERVAL_HOURS))
    ax.tick_params(axis="x", which="minor", labelbottom=False, length=3)
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center")


def plot_storm_figure(storm, config, storm_series):
    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=FIG_SIZE,
        sharex=True,
        sharey=True,
    )
    shared_time_limits = compute_shared_time_limits(storm_series)
    legend_lines = []

    for index, cycle in enumerate(CYCLES):
        panel_lines = plot_cycle_panel(
            axes[index],
            storm,
            cycle,
            config,
            storm_series[cycle],
            shared_time_limits,
        )
        format_time_axis(
            axes[index],
            config["year"],
            show_xlabel=index == len(CYCLES) - 1,
        )
        if index == 0:
            legend_lines = panel_lines

    y_min, y_max = axes[0].get_ylim()
    axes[0].set_ylim(
        y_min - LOWER_Y_PADDING_FRACTION * (y_max - y_min),
        y_max,
    )
    axes[0].tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    axes[0].legend(
        legend_lines,
        [line.get_label() for line in legend_lines],
        loc="upper left",
        frameon=True,
        fontsize=LEGEND_FONT_SIZE,
        ncol=1,
    )
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / config["output_file"],
        dpi=600,
        format="tif",
        bbox_inches="tight",
    )
    if plt.get_backend().lower() == "agg":
        plt.close(fig)
    else:
        plt.show()

In [ ]:
def validate_maxwspd_data(configs, raw_forecasts, storm_series_by_storm):
    for storm, config in configs.items():
        for cycle in CYCLES:
            expected_raw = config["expected_raw_points"][cycle]
            expected_plot = config["expected_plot_points"][cycle]
            cycle_data = storm_series_by_storm[storm][cycle]
            best_track = cycle_data["best_track"]

            assert len(best_track) == expected_plot, (
                storm, cycle, "best_track", len(best_track)
            )
            assert not best_track[["time", "wind_ms"]].isna().any().any()
            assert best_track["time"].between(
                config["init_times"][cycle], config["end_time"]
            ).all()

            for model, raw_forecast in raw_forecasts[storm][cycle].items():
                forecast = cycle_data["forecast_aligned"][model]
                assert len(raw_forecast) == expected_raw, (
                    storm, cycle, model, len(raw_forecast)
                )
                assert len(forecast) == expected_plot, (
                    storm, cycle, model, len(forecast)
                )
                assert not forecast[[
                    "time", "forecast_maxwind_ms"
                ]].isna().any().any()
                assert forecast["time"].equals(best_track["time"]), (storm, cycle, model)

            if storm == "Yagi":
                sample_forecast = cycle_data["forecast_aligned"]["era5"]
                assert sample_forecast["time"].iloc[0] == config["init_times"][cycle]
                assert sample_forecast["time"].iloc[-1] == config["end_time"]


best_tracks = {
    storm: load_best_track(config["best_track_file"])
    for storm, config in STORM_CONFIG.items()
}
raw_forecasts = {
    storm: {
        cycle: {
            model: load_maxwspd_forecast(config, storm, model, cycle)
            for model in MODEL_STYLES
        }
        for cycle in CYCLES
    }
    for storm, config in STORM_CONFIG.items()
}
storm_series_by_storm = {
    storm: {
        cycle: prepare_cycle_series(
            storm,
            cycle,
            config,
            raw_forecasts[storm][cycle],
            best_tracks[storm],
        )
        for cycle in CYCLES
    }
    for storm, config in STORM_CONFIG.items()
}
validate_maxwspd_data(STORM_CONFIG, raw_forecasts, storm_series_by_storm)

for storm in STORM_ORDER:
    plot_storm_figure(
        storm,
        STORM_CONFIG[storm],
        storm_series_by_storm[storm],
    )